In [186]:
import json
import time
from typing import Dict, List

import torch
from transformers import AutoModelForQuestionAnswering, AutoTokenizer

In [187]:
MODEL_NAME = "deepset/roberta-base-squad2"
INPUT_FILE = "../data/input.json"
OUTPUT_FILE = "../output/output.json"

TIME_LIMIT = 9.0
MAX_LEN = 384
STRIDE = 128
MAX_ANSWER_LEN = 50
TOP_K = 10
OPTIMUM_SCORE = 15

VERBOSE = 2         # 0 | 1 | 2

In [188]:
def load_data(path: str) -> List[Dict[str, str]]:
    """Load data from input file."""

    # Open the input file
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception as e:
        raise ValueError(f"Failed to read input file: {path}. Please move the file to the correct location ({path}).")

    return data

In [189]:
data = load_data(INPUT_FILE)
data

[{'question_id': 'q1',
  'question': 'What is the capital of France?',
  'document': 'France is a country in Europe. Its capital is Paris.'},
 {'question_id': 'q2',
  'question': 'Who wrote Hamlet?',
  'document': "William Shakespeare was an English playwright, widely regarded as one of the greatest writers in the English language. He wrote many famous plays, including tragedies, comedies, and histories. One of his most well-known works is the tragedy 'Hamlet', which tells the story of Prince Hamlet of Denmark and his quest for revenge."},
 {'question_id': 'q3',
  'question': 'Who is the main character?',
  'document': "Elantris is a fantasy novel written by American fantasy and science fiction writer Brandon Sanderson. The book was published on April 21, 2005, by Tor Books and was Sanderson's first published book. The book gained generally positive reviews, and has been translated into multiple languages. Two related works, Hope of Elantris and The Emperor's Soul, have been released, 

In [190]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if VERBOSE:
    print(f"Using device: {device}")

Using device: cuda


In [191]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME).to(device).eval()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 527.29it/s]
RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [192]:
def predict(sample):
    
    def time_exceed():
        return (time.perf_counter() - start_time) > TIME_LIMIT
    
    def final_answer():
        if best_answer:
            return (best_answer, best_score)
        elif best_argmax_answer:
            return (best_argmax_answer, best_argmax_score)
        else:
            return (context[:100].strip() if context.strip() else "unknown", float("-inf"))
        
    def return_answer(reason=None):
        answer, score = final_answer()
        if VERBOSE:
            if reason == "timeout":
                print(f"\033[91mTime limit of {TIME_LIMIT} seconds exceeded for question ID {question_id}.\033[0m")
            elif reason == "optimum":
                print(f"\033[94mOptimum score of {OPTIMUM_SCORE} reached. \n Answer: '{answer}' Score: {score:.3f} Time: {time.perf_counter() - start_time:.2f}s \033[0m")
            else:
                print(f"\033[92mBest answer: {answer} (score: {score:.3f}) time: {time.perf_counter() - start_time:.2f}s\033[0m")
        return {
            "question_id": question_id,
            "answer": answer
        }
    
    start_time = time.perf_counter()
    question_id = sample["question_id"]
    question = sample["question"]
    context = sample["document"]

    if VERBOSE:
        print(f"Question ID: {question_id}")
        print(f"Question: {question}")
        print(f"Document: {context[:50]}...")
    
    best_answer = ""
    best_score = float("-inf")
    best_argmax_answer = ""
    best_argmax_score = float("-inf")

    encoding = tokenizer(
        text=question,
        text_pair=context,
        padding="max_length",
        truncation="only_second",
        max_length=MAX_LEN,
        stride=STRIDE,
        return_tensors="pt",
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
    ).to(device)
    
    # Get model predictions
    with torch.no_grad():
        outputs = model(
            input_ids=encoding["input_ids"],
            attention_mask=encoding["attention_mask"]
        )
        
    start_logits = outputs.start_logits
    end_logits = outputs.end_logits

    # For each chunk
    for i, (start_logits_chunk, end_logits_chunk) in enumerate(zip(start_logits, end_logits)):
        
        # Exit search if time limit exceeded
        if time_exceed():
            return return_answer(reason="timeout")
        # Exit search if optimum score reached
        if best_score >= OPTIMUM_SCORE:
            return return_answer(reason="optimum")
        
        if VERBOSE:
            print(f"Processing chunk {i+1}/{len(start_logits)}")

        offsets = encoding["offset_mapping"][i].tolist()
        sequence_ids = encoding.sequence_ids(i)

        # Top-k start and end positions
        start_indexes = torch.topk(start_logits_chunk, TOP_K).indices.tolist()
        end_indexes = torch.topk(end_logits_chunk, TOP_K).indices.tolist()
        
        for start_idx in start_indexes:
            
            # Exit search if time limit exceeded
            if time_exceed():
                return return_answer(reason="timeout")
            # Exit search if optimum score reached
            if best_score >= OPTIMUM_SCORE:
                return return_answer(reason="optimum")
            
            
            # Skip if the start index is not part of the context
            if sequence_ids[start_idx] != 1:
                continue

            # Get character offsets for the start index
            start_char, _ = offsets[start_idx]

            for end_idx in end_indexes:
                
                # Exit search if time limit exceeded
                if time_exceed():
                    return return_answer(reason="timeout")
                # Exit search if optimum score reached
                if best_score >= OPTIMUM_SCORE:
                    return return_answer(reason="optimum")
                
                # Skip if the end index is not part of the context
                if sequence_ids[end_idx] != 1:
                    continue
                
                # Skip if start index is after end index
                if start_idx > end_idx:
                    continue
                    
                # Skip if the answer is too long
                if end_idx - start_idx + 1 > MAX_ANSWER_LEN:
                    continue

                # Get character offsets for the end index
                _, end_char = offsets[end_idx]
                
                # Skip if start_char is after end_char
                if start_char > end_char:
                    continue

                # Extract answer text from context
                answer_candidate = context[start_char:end_char].strip()
                
                # Skip if answer is empty
                if not answer_candidate:
                    continue

                # Calculate score for the candidate answer
                score_candidate = start_logits_chunk[start_idx].item() + end_logits_chunk[end_idx].item()
                
                if VERBOSE > 1:
                    print(f"Candidate answer: '{answer_candidate}', Score: {score_candidate:.3f}")
                
                # Update best answer
                if score_candidate > best_score:
                    best_score = score_candidate
                    best_answer = answer_candidate
                    
                    if VERBOSE > 1:
                        print(f"\033[93mNew best answer: '{best_answer}' with score {best_score:.3f}\033[0m")


        # Argmax fallback
        logits_sum = start_logits_chunk + end_logits_chunk
        argmax_idx = torch.argmax(logits_sum).item()
        # Skip non-context text
        if sequence_ids[argmax_idx] == 1:
            char_start, char_end = offsets[argmax_idx]
            # Skip invalid spans
            if char_start < char_end:
                argmax_score = logits_sum[argmax_idx].item()
                # Update best argmax answer
                if argmax_score > best_argmax_score:
                    best_argmax_score = argmax_score
                    best_argmax_answer = context[char_start:char_end].strip()
                
                if VERBOSE > 1:
                    print(f"\033[95mFallback answer: '{best_argmax_answer}' with score {best_argmax_score:.3f}\033[0m")                    
    
    
    answer, score = final_answer()
    
    if VERBOSE:
        print(f"\033[92mBest answer: {answer} (score: {score:.3f}) time: {time.perf_counter() - start_time:.2f}s\033[0m")

    return {
        "question_id": question_id,
        "answer": answer
    }
    

In [193]:
predictions = []

for sample in data:
    predictions.append(predict(sample))

# Save output
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(predictions, f, indent=2, ensure_ascii=False)

if VERBOSE:
    print(f"Saved predictions to {OUTPUT_FILE}")

Question ID: q1
Question: What is the capital of France?
Document: France is a country in Europe. Its capital is Pari...
Processing chunk 1/1
Candidate answer: 'Paris', Score: 12.348
New best answer: 'Paris' with score 12.348
Candidate answer: 'Paris.', Score: 7.038
Candidate answer: 'Its capital is Paris', Score: 6.413
Candidate answer: 'Its capital is Paris.', Score: 1.104
Candidate answer: 'Its capital', Score: -2.772
Candidate answer: 'Its', Score: -5.005
Candidate answer: 'Its capital is', Score: -5.476
Candidate answer: 'capital is Paris', Score: 5.372
Candidate answer: 'capital is Paris.', Score: 0.063
Candidate answer: 'capital', Score: -3.813
Candidate answer: 'capital is', Score: -6.517
Candidate answer: 'France is a country in Europe. Its capital is Paris', Score: 4.082
Candidate answer: 'France is a country in Europe. Its capital is Paris.', Score: -1.228
Candidate answer: 'France is a country in Europe. Its capital', Score: -5.104
Candidate answer: 'France', Score: -5.246
